# Mean Reciprocal Rank at K (MRR@K) — Mathematical Guide

## 1. Overview & Core Concept

**Mean Reciprocal Rank (MRR)** is an order-aware evaluation metric designed for information retrieval and recommendation systems. It evaluates systems where a user is primarily searching for a **single correct or highly relevant item**, answering the question:

> *"On average, how high up in the top $K$ recommendations does the user find their first relevant hit?"*

When evaluated with a cutoff threshold $K$ (denoted as **MRR@K**), the system only inspects the top $K$ positions. If the first relevant item appears beyond position $K$, it is treated as a complete failure (score of 0).

## 2. Mathematical Formulation

### Step 1: Reciprocal Rank at K (RR@K) for a Single Query / User
For a given query or user $u$, let $\text{rank}_u$ denote the position (1-indexed) of the **first relevant item** in the ranked list.

$$\text{RR@K}(u) = \begin{cases} 
\frac{1}{\text{rank}_u} & \text{if the first relevant item is at } \text{rank}_u \le K \\[8pt]
0 & \text{if no relevant item appears within the top } K
\end{cases}$$

* **Rank 1:** $\text{RR@K} = \frac{1}{1} = 1.000$ *(Maximum possible score)*
* **Rank 2:** $\text{RR@K} = \frac{1}{2} = 0.500$
* **Rank 3:** $\text{RR@K} = \frac{1}{3} \approx 0.333$
* **Rank 4:** $\text{RR@K} = \frac{1}{4} = 0.250$
* **Rank 5:** $\text{RR@K} = \frac{1}{5} = 0.200$
* **Rank $> K$ or Not Found:** $\text{RR@K} = 0.000$

### Step 2: Mean Reciprocal Rank at K (MRR@K) Across All Queries
Take the arithmetic mean of all individual reciprocal rank scores across the entire set of queries or users $Q$:

$$\text{MRR@K} = \frac{1}{|Q|} \sum_{u=1}^{|Q|} \text{RR@K}(u)$$

* **Range:** Strictly bounded between $[0.0, 1.0]$.
* **Optimal Score:** $\text{MRR@K} = 1.0$ (Every single user gets a relevant recommendation at position 1).

## 3. The Math Behind the Reciprocal Penalty: Why $\frac{1}{\text{rank}}$?

The reciprocal function $f(r) = \frac{1}{r}$ models **non-linear human cognitive decay and eye-tracking behavior**:

| Position Shift | Score Change | Drop in Score | Interpretation |
| :---: | :---: | :---: | :--- |
| **Rank 1 $\rightarrow$ Rank 2** | $1.000 \rightarrow 0.500$ | **$-0.500$** | **Massive penalty:** Slipping out of the #1 spot cuts the score by 50%. |
| **Rank 2 $\rightarrow$ Rank 3** | $0.500 \rightarrow 0.333$ | **$-0.167$** | Substantial penalty. |
| **Rank 3 $\rightarrow$ Rank 4** | $0.333 \rightarrow 0.250$ | **$-0.083$** | Moderate penalty. |
| **Rank 9 $\rightarrow$ Rank 10** | $0.111 \rightarrow 0.100$ | **$-0.011$** | Negligible penalty; users rarely distinguish between rank 9 and 10. |

This non-linear steep drop ensures that models are aggressively rewarded for getting items into the #1 slot.

## 4. Concrete Mathematical Example (Step-by-Step)

Suppose an e-commerce platform evaluates a recommendation model for **4 users** with a cutoff threshold of **$K = 3$**.

### User 1: Query = "Wireless Mouse"
* System Returns:
  1. `Logitech MX Master` $\rightarrow$ **[Relevant]**
  2. `Mouse Pad` $\rightarrow$ [Irrelevant]
  3. `USB Hub` $\rightarrow$ [Irrelevant]
* First relevant item appears at **$\text{rank}_1 = 1$**.
$$\text{RR@3}(1) = \frac{1}{1} = \mathbf{1.000}$$

### User 2: Query = "Python Book"
* System Returns:
  1. `Java Complete Reference` $\rightarrow$ [Irrelevant]
  2. `C++ Primer` $\rightarrow$ [Irrelevant]
  3. `Fluent Python` $\rightarrow$ **[Relevant]**
* First relevant item appears at **$\text{rank}_2 = 3$**.
$$\text{RR@3}(2) = \frac{1}{3} \approx \mathbf{0.333}$$

### User 3: Query = "Running Shoes"
* System Returns:
  1. `Water Bottle` $\rightarrow$ [Irrelevant]
  2. `Nike Air Zoom` $\rightarrow$ **[Relevant]**
  3. `Adidas Ultraboost` $\rightarrow$ **[Relevant]**
* First relevant item appears at **$\text{rank}_3 = 2$**.
* *Note: Rank 3 is also relevant, but MRR ignores all hits after the first one.*
$$\text{RR@3}(3) = \frac{1}{2} = \mathbf{0.500}$$

### User 4: Query = "Electric Scooter"
* System Returns:
  1. `Bicycle Helmet` $\rightarrow$ [Irrelevant]
  2. `Bike Lock` $\rightarrow$ [Irrelevant]
  3. `Skateboard` $\rightarrow$ [Irrelevant]
* No relevant item is present in the top 3 (even if an electric scooter was at rank 4).
$$\text{RR@3}(4) = \mathbf{0.000}$$

### Final Calculation of MRR@3:

$$\text{MRR@3} = \frac{\text{RR@3}(1) + \text{RR@3}(2) + \text{RR@3}(3) + \text{RR@3}(4)}{4}$$

$$\text{MRR@3} = \frac{1.000 + 0.333 + 0.500 + 0.000}{4}$$

$$\text{MRR@3} = \frac{1.833}{4} = \mathbf{0.458} \quad (45.8\%)$$

## 5. Python Implementation

```python
import numpy as np

def compute_mrr_at_k(recommended_lists, ground_truth_sets, k=10):
    """
    Computes Mean Reciprocal Rank at K (MRR@K).
    
    Parameters:
    - recommended_lists: list of lists, where each inner list contains ordered item IDs.
    - ground_truth_sets: list of sets, where each inner set contains relevant item IDs for that user.
    - k: cutoff rank threshold.
    
    Returns:
    - float: MRR@K score between 0.0 and 1.0.
    """
    reciprocal_ranks = []
    
    for rec_list, true_set in zip(recommended_lists, ground_truth_sets):
        # Truncate to top K
        top_k = rec_list[:k]
        rr = 0.0
        
        for rank, item in enumerate(top_k, start=1):
            if item in true_set:
                rr = 1.0 / rank
                break  # Stop at the FIRST relevant item!
                
        reciprocal_ranks.append(rr)
        
    return np.mean(reciprocal_ranks)

# Example Usage:
recs = [
    ['item_A', 'item_B', 'item_C'],  # User 1
    ['item_X', 'item_Y', 'item_Z'],  # User 2
    ['item_M', 'item_N', 'item_P'],  # User 3
    ['item_1', 'item_2', 'item_3']   # User 4
]

truth = [
    {'item_A'},             # Hit at Rank 1 -> RR = 1.0
    {'item_Z'},             # Hit at Rank 3 -> RR = 0.333
    {'item_N', 'item_P'},   # First hit at Rank 2 -> RR = 0.5
    {'item_9'}              # No hit -> RR = 0.0
]

score = compute_mrr_at_k(recs, truth, k=3)
print(f"MRR@3: {score:.3f}")  # Output: 0.458
```

## 6. When and Where to Use MRR@K

* **Search Engines & Navigational Search:** When users are looking for a single specific webpage or document (e.g., searching for "Google Flights").
* **Voice Assistants (Alexa, Siri, Google Assistant):** When the AI agent can only speak **one single answer** aloud.
* **Auto-complete & Typeahead Search Bars:** In search dropdowns, user friction drops dramatically if the intended query is the very 1st auto-complete suggestion.
* **E-Commerce Direct Reorders:** Predicting the exact grocery item the user wants to reorder today.

## 7. Pros and Cons of MRR@K

### Pros:
1. **Directly Models the #1 Spot:** Heavily penalizes systems that miss the top spot, encouraging high top-tier precision.
2. **Simple and Computationally Fast:** $O(K)$ lookup per user; stops scanning the list immediately after the first match.
3. **Bounded & Standardized:** Always outputs a clean score between $0.0$ and $1.0$, allowing fair comparison across diverse models.

### Cons:
1. **Blind to Subsequent Hits:** If User 3 likes items at positions 2, 3, 4, and 5, MRR strictly scores position 2 and throws away the other three hits.
2. **Binary Relevance Only:** Treats all relevant items as equally good; cannot handle graded feedback (e.g., 5-star vs 3-star).
3. **Inappropriate for Exploratory Browsing:** Poorly suited for Netflix or Spotify home feeds, where users want a diverse list of multiple good options.

## 8. Summary Comparison: MRR@K vs. MAP@K vs. NDCG@K

| Metric | Target Relevance Type | Considers Hits After 1st? | Position Decay Function | Primary Use Case |
| :--- | :---: | :---: | :---: | :--- |
| **MRR@K** | Binary (0 / 1) | **No** (First hit only) | Reciprocal: $\frac{1}{r}$ | Question Answering, Search, Voice Assistants |
| **MAP@K** | Binary (0 / 1) | **Yes** (All hits in top $K$) | Precision at each rank | Catalogs, Multiple Binary Recommendations |
| **NDCG@K**| **Graded** (0 to 5 stars) | **Yes** (All hits in top $K$) | Logarithmic: $\frac{1}{\log_2(r+1)}$ | Home Feeds (Netflix, Spotify, Amazon) |

---

##  **Why do we use MRR, i think RR itself is sufficient enough to tell me how my recommendation engine is working ?** 

#### **The Fundamental Difference**
- RR (Reciprocal Rank): Evaluates ONE single user or ONE search query.
- MRR (Mean Reciprocal Rank): Evaluates the entire recommendation engine across all users.